# 🐛 Pest Detection Training with YOLOv8

This notebook provides a complete pipeline for training a YOLOv8 model for pest detection on Google Colab.

## 📋 Setup Instructions for Google Colab:
1. **Upload your dataset** to Google Drive in the following structure:
   ```
   /content/drive/MyDrive/pest_detection/
   ├── datasets/
   │   └── yoloip1/
   │       ├── data.yaml
   │       ├── train/
   │       ├── valid/
   │       └── test/
   ```
2. **Run all cells sequentially** from top to bottom
3. **GPU Runtime**: Go to `Runtime → Change runtime type → GPU (T4 or better)`
4. Training will take 2-4 hours depending on your GPU

## 🎯 What This Notebook Does:
- Installs required dependencies (ultralytics, opencv, etc.)
- Mounts your Google Drive
- Configures the dataset paths
- Trains a YOLOv8 model on your pest detection dataset
- Saves the best model for deployment


## 1. Install Dependencies

Install the required Python packages for YOLOv8 training.


In [ ]:
%pip install ultralytics opencv-python-headless pillow pyyaml -q
print("✅ Dependencies installed successfully!")


## 2. Mount Google Drive

Mount your Google Drive to access the dataset.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted successfully!")


## 3. Configure Paths

Set up the paths to your dataset. **IMPORTANT**: Update `BASE_PATH` to match where you uploaded your dataset in Google Drive.


In [ ]:
import os
from pathlib import Path

# ⚠️ CHANGE THIS PATH to match your Google Drive folder structure
BASE_PATH = '/content/drive/MyDrive/pest_detection'
DATASET_PATH = f'{BASE_PATH}/datasets/yoloip1'
DATA_YAML = f'{DATASET_PATH}/data.yaml'

# Verify paths exist
if not os.path.exists(DATASET_PATH):
    print(f"❌ Dataset not found at: {DATASET_PATH}")
    print("Please upload your dataset to Google Drive and update the BASE_PATH variable above.")
else:
    print(f"✅ Dataset found at: {DATASET_PATH}")
    
if not os.path.exists(DATA_YAML):
    print(f"❌ data.yaml not found at: {DATA_YAML}")
else:
    print(f"✅ data.yaml found")


## 4. Fix data.yaml Paths

Update the data.yaml file to use absolute paths required by YOLOv8.


In [ ]:
import yaml

def fix_data_yaml():
    """Fix the data.yaml file paths to be absolute"""
    # Read the current yaml file
    with open(DATA_YAML, 'r') as f:
        data = yaml.safe_load(f)
    
    # Update paths to be absolute
    data['train'] = os.path.join(DATASET_PATH, 'train', 'images')
    data['val'] = os.path.join(DATASET_PATH, 'valid', 'images')
    data['test'] = os.path.join(DATASET_PATH, 'test', 'images')
    
    # Write the updated yaml file
    with open(DATA_YAML, 'w') as f:
        yaml.dump(data, f, default_flow_style=False)
    
    print(f"✅ Updated data.yaml with absolute paths")
    print(f"  Train: {data['train']}")
    print(f"  Val: {data['val']}")
    print(f"  Test: {data['test']}")
    print(f"  Classes: {data['names']}")
    
    return data

# Fix the data.yaml
dataset_info = fix_data_yaml()


## 5. Check GPU Availability

Verify that you're using a GPU runtime for faster training.


In [ ]:
import torch

if torch.cuda.is_available():
    device = 'cuda'
    gpu_name = torch.cuda.get_device_name(0)
    print(f"✅ GPU available: {gpu_name}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    device = 'cpu'
    print("⚠️  WARNING: GPU not available, using CPU")
    print("   Training will be VERY slow. Consider enabling GPU in Runtime → Change runtime type")

print(f"   Device: {device}")


## 6. Download Pre-trained YOLOv8 Model

Download the YOLOv8 nano model as a starting point for training.


In [ ]:
from ultralytics import YOLO

# Load a pre-trained YOLOv8 model (it will auto-download if not present)
print("Loading YOLOv8n pre-trained model...")
model = YOLO('yolov8n.pt')  # Using nano for faster training
print("✅ Model loaded successfully!")


## 7. Training Configuration

Configure the training parameters. You can adjust these based on your needs:
- `epochs`: Number of training iterations (50-100 recommended)
- `imgsz`: Image size (640 for better accuracy, 416 for faster training)
- `batch`: Batch size (adjust based on GPU memory)
- `device`: 'cuda' for GPU or 'cpu' (auto-detected above)


In [ ]:
# Training configuration
TRAINING_CONFIG = {
    'data': DATA_YAML,
    'epochs': 100,        # Number of training epochs (increase for better accuracy)
    'imgsz': 640,         # Image size (640 for better accuracy, 416 for faster training)
    'batch': 16,          # Batch size (adjust based on GPU memory: 16 for T4, 32 for A100)
    'device': device,     # Auto-detected GPU or CPU
    'workers': 2,         # Number of data loading workers
    'project': f'{BASE_PATH}/runs/train',  # Save results to Google Drive
    'name': 'pest_detection',  # Name of the training run
    'exist_ok': True,     # Overwrite existing runs
    'patience': 15,       # Early stopping patience
    'save': True,         # Save checkpoints
    'save_period': 10,    # Save every 10 epochs
    'cache': True if device == 'cuda' else False,  # Cache images for faster training (GPU only)
    'verbose': True       # Verbose output
}

print("Training Configuration:")
for key, value in TRAINING_CONFIG.items():
    print(f"  {key}: {value}")


## 8. Start Training 🚀

This cell will start the training process. This typically takes 2-4 hours with a GPU.

**What to expect:**
- Training progress bar with loss metrics
- Validation metrics after each epoch
- Model checkpoints saved to Google Drive
- The best model will be saved automatically


In [ ]:
print("="*60)
print("🚀 STARTING YOLOIP1 PEST DETECTION MODEL TRAINING")
print("="*60)
print()

try:
    # Train the model
    results = model.train(**TRAINING_CONFIG)
    
    print("\n" + "="*60)
    print("✅ TRAINING COMPLETED SUCCESSFULLY!")
    print("="*60)
    
except Exception as e:
    print(f"\n❌ Training failed: {str(e)}")
    print("\nTroubleshooting tips:")
    print("1. Check if you have enough GPU memory (reduce batch size if needed)")
    print("2. Verify dataset paths are correct")
    print("3. Ensure all dataset files are properly formatted")
    print("4. Try reducing image size or batch size")


## 9. Copy Best Model

Copy the best trained model to your dataset directory for easy access.


In [ ]:
import shutil

# Define paths
best_model_path = f'{BASE_PATH}/runs/train/pest_detection/weights/best.pt'
target_path = f'{DATASET_PATH}/best.pt'

# Copy the best model
if os.path.exists(best_model_path):
    shutil.copy2(best_model_path, target_path)
    model_size = os.path.getsize(target_path) / (1024 * 1024)  # Size in MB
    print(f"✅ Best model copied to: {target_path}")
    print(f"   Model size: {model_size:.2f} MB")
    print("\n📥 Download this file to use in your pest detection system!")
else:
    print(f"❌ Best model not found at: {best_model_path}")
    print("   Check the training output above for errors.")


## 10. View Training Results

Visualize the training results including loss curves and performance metrics.


In [ ]:
from IPython.display import Image, display
import glob

results_dir = f'{BASE_PATH}/runs/train/pest_detection'

# Display training results
result_images = [
    'results.png',
    'confusion_matrix.png',
    'labels.jpg',
    'train_batch0.jpg',
    'val_batch0_pred.jpg'
]

print("📊 Training Results:\n")
for img_name in result_images:
    img_path = f'{results_dir}/{img_name}'
    if os.path.exists(img_path):
        print(f"--- {img_name} ---")
        display(Image(filename=img_path, width=800))
    else:
        print(f"⚠️  {img_name} not found")


## 11. Test the Model (Optional)

Test the trained model on sample images to verify it's working correctly.


In [ ]:
# Load the trained model
trained_model = YOLO(target_path)

# Get a few test images
test_images_dir = f'{DATASET_PATH}/test/images'
test_images = glob.glob(f'{test_images_dir}/*.jpg')[:5]  # Get first 5 test images

if test_images:
    print("🔍 Testing model on sample images:\n")
    for img_path in test_images:
        # Run inference
        results = trained_model(img_path, conf=0.4)
        
        # Display the result
        result_img = results[0].plot()
        
        # Convert BGR to RGB for proper display
        import cv2
        result_img_rgb = cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB)
        
        print(f"Image: {os.path.basename(img_path)}")
        display(Image(data=cv2.imencode('.jpg', result_img_rgb)[1].tobytes(), width=600))
        print()
else:
    print("⚠️  No test images found")


## 🎉 Training Complete!

Your pest detection model has been trained successfully. Here's what to do next:

### 📥 Download Your Model:
1. Go to your Google Drive at: `{BASE_PATH}/datasets/yoloip1/best.pt`
2. Download the `best.pt` file

### 🚀 Deploy Your Model:
1. Replace the `best.pt` file in your local project
2. Run your pest detection system using `object_detection.py`
3. The model will now detect pests in real-time!

### 📊 Training Artifacts:
All training results are saved in: `{BASE_PATH}/runs/train/pest_detection/`
- `weights/`: Model checkpoints
- `results.png`: Training metrics graphs
- `confusion_matrix.png`: Model performance visualization
- `results.csv`: Detailed training metrics

---

**Need to retrain?** Just run this notebook again with different parameters!
